# SWE015 — Küçük modele bilgi damıtma
Önceki BERT öğretmenimiz hazır. Bu defter BERT-Mini öğrencisini gerçek etiketler
ve öğretmenin yumuşatılmış sınıf tahminleriyle eğitir. Test verisi kullanılmaz.
**Runtime → Change runtime type → T4 GPU** seç; hücreleri sırayla çalıştır.
Bu aşama sonunda FP32 öğrenci indirilir. INT8 quantization ve son test bilgisayarında ayrıca yapılacak.


## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), "T4 GPU sec ve tekrar baglan."
print("GPU:", torch.cuda.get_device_name(0))


## 2. Yeni eğitim paketini yükle
`colab/student_training_bundle.zip` dosyasını seç. Eski BERT paketini seçme.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, json, hashlib
uploaded = files.upload()
assert len(uploaded) == 1, "Tek ZIP sec."
ROOT = Path("/content/support_router_student")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for entry in archive.infolist():
        assert (ROOT / entry.filename).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
manifest = json.loads((ROOT / "colab_manifest.json").read_text())
for name, digest in manifest["files"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest, name
print("Paket dogrulandi.")


## 3. Kütüphaneler
GPU destekli mevcut PyTorch kullanılır.

In [ ]:
import subprocess, sys
def run_visible(command, log_path):
    with open(log_path, "w", encoding="utf-8") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Islem durdu ({status}). Yukaridaki hata mesajini gonder. Log: {log_path}")
run_visible([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-colab.txt")], "/content/student_install.log")
run_visible([sys.executable, str(ROOT / "scripts/train_bert.py"), "--check-data"], "/content/student_data.log")


## 4. Drive'daki öğretmeni bağla
Önceki eğitimin model klasörü otomatik seçilmiştir. Dosyalar aynı BERT olduğundan emin olmak için doğrulanır.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount("/content/drive")
TEACHER = Path('/content/drive/MyDrive/SWE015_Customer_Support_Router/bert_20261119_174139/bert_model')
assert (TEACHER / "model.safetensors").exists(), "Ogretmen klasoru bulunamadi. Yolu kontrol et."
RUN = Path("/content/drive/MyDrive/SWE015_Customer_Support_Router") / ("student_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))
print("Ogretmen:", TEACHER)
print("Ogrenci kayit klasoru:", RUN)


## 5. Öğrenciyi eğit
Önce öğretmen yalnızca eğitim mesajlarını puanlar; sonra 6 epoch öğrenci eğitilir.
Kayıp = %50 gerçek etiket kaybı + %50 yumuşak öğretmen dağılımı kaybı; sıcaklık 2.
En iyi model doğrulama kaybıyla seçilir. Güven eşiği de yalnızca doğrulamayla belirlenir.
Çalışırken yeniden başlatma. Hata olursa buradaki son mesajı sohbete gönder.
Drive'a model checkpoint'leri kaydedilir; optimizer saklanmadığından otomatik adım devamı yoktur.


In [ ]:
run_visible([sys.executable, "-u", str(ROOT / "scripts/train_student.py"),
             "--teacher", str(TEACHER), "--output-dir", str(RUN)], "/content/student_training.log")


## 6. Sonuç ve indirme
Bunlar doğrulama ölçümleridir; test sonucu değildir. İnen `student_model.zip` dosyasını projenin `colab/` klasörüne koy.

In [ ]:
import shutil
assert (RUN / "student_model/reload_check.json").exists(), "Yukleme kontrolu tamamlanmadi; once hata mesajini inceleyelim."
report = json.loads((RUN / "student_model/training_report.json").read_text())
print("Dogrulama:", report["validation_selected"])
print("Parametre sayisi:", report["parameters"])
archive = shutil.make_archive("/content/student_model", "zip", root_dir=RUN, base_dir="student_model")
shutil.copy2(archive, RUN / "student_model.zip")
shutil.copy2("/content/student_training.log", RUN / "student_training.log")
files.download(archive)


Kaynak: [Google BERT-Mini](https://huggingface.co/google/bert_uncased_L-4_H-256_A-4).
Öğretmen bu projede eğitilen BERT'tir. Kod hazırlığında Codex kullanıldı.
Sonraki aşama INT8 quantization ve tüm modellerin aynı testte karşılaştırılmasıdır.
